# ASL Fingerspelling — Training Notebook

Trains a Random Forest classifier on the Kaggle ASL alphabet dataset.

**Feature vector**: 78 features per sample
- 63 normalized (x, y, z) landmarks (21 points x 3)
- 15 finger joint angles (3 joints x 5 fingers)

**Output**: `classifier/classify_letter_model.p`

> Uses the MediaPipe Tasks API (`mp.tasks.vision.HandLandmarker`).
> The old `mp.solutions.hands` API was removed in MediaPipe 0.10.x.

## Imports & Config

In [ ]:
import os
import cv2
import pickle
import random
import urllib.request
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Only import pure-numpy helpers from utils — extract_landmarks uses the old API
from utils import build_feature_vector

# New Tasks API aliases
HandLandmarker        = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
RunningMode           = mp.tasks.vision.RunningMode
BaseOptions           = mp.tasks.BaseOptions


In [ ]:
# Paths
TRAIN_DIR    = './data/asl_alphabet_train/asl_alphabet_train'
DATASET_PATH = './data/letter_dataset.pickle'
MODEL_PATH   = './classifier/classify_letter_model.p'
LANDMARKER_MODEL = './hand_landmarker.task'   # downloaded below

# How many images to sample per class.
# 3000 images/class x 29 classes = 87k total — too slow to process all.
# 500/class gives ~14.5k samples and processes in ~5 min on CPU.
# Set to None to use every image.
SAMPLES_PER_CLASS = 500
RANDOM_SEED = 42
MIN_DETECTION_CONFIDENCE = 0.5

ALL_CLASSES = sorted(os.listdir(TRAIN_DIR))
print(f'Found {len(ALL_CLASSES)} class folders: {ALL_CLASSES}')


## Download Hand Landmarker Model

The new Tasks API requires a `.task` bundle — download it once.

In [ ]:
LANDMARKER_URL = (
    'https://storage.googleapis.com/mediapipe-models/'
    'hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task'
)

if not os.path.exists(LANDMARKER_MODEL):
    print('Downloading hand_landmarker.task ...')
    urllib.request.urlretrieve(LANDMARKER_URL, LANDMARKER_MODEL)
    print(f'Saved -> {LANDMARKER_MODEL}')
else:
    print(f'Model already present: {LANDMARKER_MODEL}')


## Landmark Extraction Helper

Wrapper around the new Tasks API that returns the same 63-float format
as the old `extract_landmarks()` in utils.py.

In [ ]:
def extract_landmarks_new(result):
    """
    Extract 63 floats (21 landmarks x xyz) from a HandLandmarker result.
    Returns None if no hand was detected.

    Tasks API result structure (different from old solutions API):
        result.hand_landmarks  -> list[list[NormalizedLandmark]]
        result.hand_landmarks[0] -> 21 landmarks for the first hand
    """
    if not result.hand_landmarks:
        return None
    coords = []
    for lm in result.hand_landmarks[0]:   # first detected hand
        coords.extend([lm.x, lm.y, lm.z])
    return coords   # 63 floats


## Build Dataset

Process each image through MediaPipe -> extract 78-feature vector.
Skips images where MediaPipe fails to detect a hand.

In [ ]:
options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=LANDMARKER_MODEL),
    running_mode=RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=MIN_DETECTION_CONFIDENCE
)

data    = []
labels  = []
skipped = 0

with HandLandmarker.create_from_options(options) as detector:
    for class_name in ALL_CLASSES:
        class_dir   = os.path.join(TRAIN_DIR, class_name)
        image_files = os.listdir(class_dir)

        if SAMPLES_PER_CLASS is not None and len(image_files) > SAMPLES_PER_CLASS:
            random.seed(RANDOM_SEED)
            image_files = random.sample(image_files, SAMPLES_PER_CLASS)

        class_added = 0
        for fname in image_files:
            img_path = os.path.join(class_dir, fname)
            img = cv2.imread(img_path)
            if img is None:
                skipped += 1
                continue

            img_rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
            result   = detector.detect(mp_image)

            raw = extract_landmarks_new(result)   # 63 floats or None
            if raw is None:
                skipped += 1
                continue

            features = build_feature_vector(raw, use_angles=True)   # 78 floats
            data.append(features)
            labels.append(class_name)
            class_added += 1

        print(f'  {class_name:20s}: {class_added} samples')

print(f'\nTotal samples             : {len(data)}')
print(f'Skipped (no hand detected): {skipped}')
print(f'Feature vector size       : {len(data[0])}')


In [ ]:
with open(DATASET_PATH, 'wb') as f:
    pickle.dump({'data': data, 'labels': labels}, f)

print(f'Dataset saved -> {DATASET_PATH}')


## Train Classifier

Load the saved dataset and train a Random Forest.

In [ ]:
with open(DATASET_PATH, 'rb') as f:
    dataset = pickle.load(f)

X = np.array(dataset['data'],   dtype=np.float32)
y = np.array(dataset['labels'])

print(f'X shape : {X.shape}')
print(f'y shape : {y.shape}')
print(f'Classes : {sorted(set(y))}')


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, stratify=y, random_state=RANDOM_SEED
)

clf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc    = accuracy_score(y_test, y_pred)
print(f'Test accuracy: {acc:.2%}')


In [ ]:
with open(MODEL_PATH, 'wb') as f:
    pickle.dump({'model': clf}, f)

print(f'Model saved -> {MODEL_PATH}')


## Evaluation

In [ ]:
print(classification_report(y_test, y_pred, digits=4))


In [ ]:
label_order = sorted(set(y))
cm = confusion_matrix(y_test, y_pred, labels=label_order)

fig, ax = plt.subplots(figsize=(14, 14))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_order)
disp.plot(cmap=plt.cm.Blues, values_format='g', ax=ax)
plt.title('Confusion Matrix — ASL Letter Classifier')
plt.tight_layout()
plt.show()


## Hard Cases

Shows which letter pairs the model confuses most. Common hard pairs: A/S/E, U/V, M/N.

In [ ]:
cm_copy = cm.copy()
np.fill_diagonal(cm_copy, 0)

top_n      = 10
flat_idx   = np.argsort(cm_copy.ravel())[::-1][:top_n]
rows, cols = np.unravel_index(flat_idx, cm_copy.shape)

print(f'Top {top_n} most confused pairs (true -> predicted : count):')
for r, c in zip(rows, cols):
    if cm_copy[r, c] > 0:
        print(f'  {label_order[r]:12s} -> {label_order[c]:12s} : {cm_copy[r, c]}')


## Quick Sanity Check

Run the saved model on the held-out test images.

In [ ]:
TEST_DIR   = './data/asl_alphabet_test/asl_alphabet_test'
test_files = sorted(f for f in os.listdir(TEST_DIR) if f.endswith('.jpg'))

with HandLandmarker.create_from_options(options) as detector:
    for fname in test_files:
        img_path   = os.path.join(TEST_DIR, fname)
        true_label = fname.replace('_test.jpg', '').upper()

        img      = cv2.imread(img_path)
        img_rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
        result   = detector.detect(mp_image)

        raw = extract_landmarks_new(result)
        if raw is None:
            print(f'  ??  true={true_label:10s}  (no hand detected)')
            continue

        features = build_feature_vector(raw, use_angles=True)
        pred     = clf.predict([features])[0]
        ok       = pred.upper() == true_label
        print(f"  {'OK' if ok else 'XX'}  true={true_label:10s}  pred={pred}")
